### Tester l'API directement dans Colab

In [21]:
!git clone https://github.com/antoinechemavonian/futurisys-api.git

fatal: destination path 'futurisys-api' already exists and is not an empty directory.


In [24]:
# 1. On installe les outils
!pip install fastapi uvicorn joblib pandas scikit-learn nest_asyncio

# 2. On se place dans le bon dossier contenant l'application
import os
import sys

# On s'assure d'être dans le dossier du dépôt cloné contenant le dossier 'app'
if os.path.exists('/content/futurisys-api'):
    os.chdir('/content/futurisys-api')
    if '/content/futurisys-api' not in sys.path:
        sys.path.insert(0, '/content/futurisys-api')
else:
    os.chdir('/content')

# Force le rechargement du module app.main s'il a déjà été importé
for mod in list(sys.modules.keys()):
    if mod.startswith('app'):
        del sys.modules[mod]

# 3. Lancement du serveur dans un thread séparé avec sa propre boucle d'événements
import threading
import asyncio
import uvicorn
from app.main import app

def start_api_server():
    # On crée et configure une nouvelle boucle d'événements dédiée à ce thread
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="info")
    server = uvicorn.Server(config)

    # On exécute le serveur sur cette boucle d'événements
    loop.run_until_complete(server.serve())

print("Lancement de l'API en arrière-plan...")
thread = threading.Thread(target=start_api_server, daemon=True)
thread.start()

Lancement de l'API en arrière-plan...


In [26]:
import requests
import json

# On teste la route /health
print("Test de la route /health :")
try:
    reponse_health = requests.get("http://127.0.0.1:8000/health")
    print("Statut :", reponse_health.status_code)
    print("Réponse :", reponse_health.json())
except Exception as e:
    print("Erreur lors de l'appel à /health :", e)

print("\n--- Test de la route /predict ---")
# On envoie un dictionnaire d'employé.
# NOTE : Si le modèle a besoin d'autres variables, il faudra les ajouter ici.
donnees_employe = {
    "age": 35,
    "revenu_mensuel": 5000.0,
    "annees_dans_l_entreprise": 5,
    "annee_experience_totale": 10,                 # Exemple de variable requise
    "annees_dans_le_poste_actuel": 3,              # Exemple de variable requise
    "annees_depuis_la_derniere_promotion": 1,      # Exemple de variable requise
    "annees_sous_responsable_actuel": 3,           # Exemple de variable requise
    "augmentation_salaire_precedente": 15          # Exemple de variable requise
}

try:
    reponse_predict = requests.post("http://127.0.0.1:8000/predict", json=donnees_employe)
    print("Statut :", reponse_predict.status_code)
    try:
        print("Réponse :", reponse_predict.json())
    except json.JSONDecodeError:
        print("La réponse n'est pas au format JSON valide. Contenu de la réponse :")
        print(reponse_predict.text)
except Exception as e:
    print("Erreur lors de la requête :", e)

Test de la route /health :
INFO:     127.0.0.1:43324 - "GET /health HTTP/1.1" 200 OK
Statut : 200
Réponse : {'status': 'ok', 'modele_charge': True}

--- Test de la route /predict ---
INFO:     127.0.0.1:43326 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Statut : 422
Réponse : {'detail': [{'type': 'missing', 'loc': ['body', 'data'], 'msg': 'Field required', 'input': {'age': 35, 'revenu_mensuel': 5000.0, 'annees_dans_l_entreprise': 5, 'annee_experience_totale': 10, 'annees_dans_le_poste_actuel': 3, 'annees_depuis_la_derniere_promotion': 1, 'annees_sous_responsable_actuel': 3, 'augmentation_salaire_precedente': 15}}]}


In [28]:
# Affichons le contenu du fichier app/main.py pour comprendre comment l'API traite les données
with open('/content/futurisys-api/app/main.py', 'r') as f:
    print(f.read())

from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd
from typing import Dict, Any
from sqlalchemy import create_engine, Column, Integer, String, Float, JSON
from sqlalchemy.orm import declarative_base, sessionmaker
import os

# --- CHEMINS INTELLIGENTS (marche partout) ---
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
MODEL_PATH = os.path.join(BASE_DIR, "modele_attrition.joblib")
DB_PATH = os.path.join(BASE_DIR, "predictions.db")

# --- CONFIGURATION BASE DE DONNÉES ---
engine = create_engine(f"sqlite:///{DB_PATH}", connect_args={"check_same_thread": False})
Base = declarative_base()

class PredictionLog(Base):
    __tablename__ = "prediction_logs"
    id = Column(Integer, primary_key=True, index=True)
    input_data = Column(JSON)
    prediction = Column(String)
    probability = Column(Float)

Base.metadata.create_all(bind=engine)
SessionLocal = sessionmaker(bind=engine)

# --- CHARGEMENT DU MODÈLE ---
objets = joblib.l

In [30]:
%%writefile /content/futurisys-api/app/main.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd

# Chargement du modèle et de ses métadonnées
objets = joblib.load("modele_attrition.joblib")
modele = objets["modele"]
colonnes = objets["colonnes"]
seuil = objets["seuil"]

app = FastAPI(title="API Attrition TechNova")

# Définition de ce que l'API attend en entrée
class EmployeInput(BaseModel):
    age: int
    revenu_mensuel: float
    annees_dans_l_entreprise: int

@app.get("/health")
def health():
    return {"status": "ok", "modele_charge": True}

@app.post("/predict")
def predict(employe: EmployeInput):
    # 1. On crée un dictionnaire avec toutes les colonnes requises initialisées à 0
    donnees_completes = {col: 0 for col in colonnes}

    # 2. On injecte les valeurs reçues via l'API
    donnees_completes["age"] = employe.age
    donnees_completes["revenu_mensuel"] = employe.revenu_mensuel
    donnees_completes["annees_dans_l_entreprise"] = employe.annees_dans_l_entreprise

    # 3. On crée le DataFrame avec la structure exacte attendue par le modèle
    input_data = pd.DataFrame([donnees_completes])[colonnes]

    # Prédiction
    proba = modele.predict_proba(input_data)[:, 1][0]
    prediction = "Oui" if proba >= seuil else "Non"

    return {"prediction": prediction, "probabilite": float(proba)}

Overwriting /content/futurisys-api/app/main.py


In [31]:
import requests
import json

# 1. Test de la route /health
print("Test de la route /health :")
try:
    reponse_health = requests.get("http://127.0.0.1:8000/health")
    print("Statut :", reponse_health.status_code)
    print("Réponse :", reponse_health.json())
except Exception as e:
    print("Erreur lors de l'appel à /health :", e)

print("\n--- Test de la route /predict ---")
# 2. Test de la route /predict avec les données de l'employé
donnees_employe = {
    "age": 35,
    "revenu_mensuel": 5000.0,
    "annees_dans_l_entreprise": 5
}

try:
    reponse_predict = requests.post("http://127.0.0.1:8000/predict", json=donnees_employe)
    print("Statut :", reponse_predict.status_code)
    try:
        print("Réponse :", reponse_predict.json())
    except json.JSONDecodeError:
        print("La réponse n'est pas au format JSON valide. Contenu de la réponse :")
        print(reponse_predict.text)
except Exception as e:
    print("Erreur lors de la requête :", e)

Test de la route /health :
INFO:     127.0.0.1:46012 - "GET /health HTTP/1.1" 200 OK
Statut : 200
Réponse : {'status': 'ok', 'modele_charge': True}

--- Test de la route /predict ---
INFO:     127.0.0.1:46028 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Statut : 422
Réponse : {'detail': [{'type': 'missing', 'loc': ['body', 'data'], 'msg': 'Field required', 'input': {'age': 35, 'revenu_mensuel': 5000.0, 'annees_dans_l_entreprise': 5}}]}


### Test toutes les colonnes

In [32]:
import requests

# On prépare des données avec TOUTES les colonnes attendues par le modèle
donnees_employe = {
    "data": {
        "age": 35,
        "revenu_mensuel": 5000.0,
        "annees_dans_l_entreprise": 5,
        "annee_experience_totale": 10,
        "annees_dans_le_poste_actuel": 2,
        "annees_depuis_la_derniere_promotion": 1,
        "annees_sous_responsable_actuel": 3,
        "augmentation_salaire_precedente": 15.0,
        "nombre_experiences_precedentes": 2,
        "niveau_hierarchique_poste": 2,
        "note_evaluation_precedente": 3,
        "note_evaluation_actuelle": 3,
        "satisfaction_employee_environnement": 3,
        "satisfaction_employee_nature_travail": 3,
        "satisfaction_employee_equipe": 3,
        "satisfaction_employee_equilibre_pro_perso": 3,
        "nombre_participation_pee": 1,
        "nb_formations_suivies": 2,
        "distance_domicile_travail": 10,
        "niveau_education": 3
        # Note : si ton modèle attend des colonnes en plus (ex: genre, département),
        # le code ajoutera automatiquement 0, ce qui permettra au modèle de faire la prédiction.
    }
}

print("Test de la route /predict avec toutes les données...")
reponse_predict = requests.post("http://127.0.0.1:8000/predict", json=donnees_employe)

print("Statut :", reponse_predict.status_code)
print("Réponse :", reponse_predict.json())

Test de la route /predict avec toutes les données...
INFO:     127.0.0.1:46032 - "POST /predict HTTP/1.1" 200 OK
Statut : 200
Réponse : {'prediction': 'Non', 'probabilite': 0.18158140466034747}


# PostgreSQL Base de données

In [33]:
!pip install sqlalchemy
# installation de SQL alchemy

## Etape 2 Modifier main.py pour ajouter la base de données

In [35]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import pandas as pd
from typing import Dict, Any
from sqlalchemy import create_engine, Column, Integer, String, Float, JSON
from sqlalchemy.orm import declarative_base, sessionmaker

# --- CONFIGURATION BASE DE DONNÉES ---
#engine = create_engine("sqlite:///predictions.db", connect_args={"check_same_thread": False})
engine = create_engine("sqlite:////content/predictions.db", connect_args={"check_same_thread": False})
Base = declarative_base()

class PredictionLog(Base):
    __tablename__ = "prediction_logs"
    id = Column(Integer, primary_key=True, index=True)
    input_data = Column(JSON)
    prediction = Column(String)
    probability = Column(Float)

Base.metadata.create_all(bind=engine)
SessionLocal = sessionmaker(bind=engine)

# --- CHARGEMENT DU MODÈLE ---
objets = joblib.load("modele_attrition.joblib")
modele = objets["modele"]
colonnes = objets["colonnes"]
seuil = objets["seuil"]

app = FastAPI(title="API Attrition TechNova")

class EmployeInput(BaseModel):
    data: Dict[str, Any]

@app.get("/health")
def health():
    return {"status": "ok", "modele_charge": True}

@app.post("/predict")
def predict(employe: EmployeInput):
    input_dict = employe.data
    input_data = pd.DataFrame([input_dict])

    for col in colonnes:
        if col not in input_data.columns:
            input_data[col] = 0

    input_data = input_data[colonnes]

    proba = modele.predict_proba(input_data)[:, 1][0]
    prediction = "Oui" if proba >= seuil else "Non"

    # --- SAUVEGARDE EN BASE DE DONNÉES ---
    db = SessionLocal()
    log = PredictionLog(
        input_data=input_dict,
        prediction=prediction,
        probability=float(proba)
    )
    db.add(log)
    db.commit()
    db.close()

    return {"prediction": prediction, "probabilite": float(proba)}

## étape 3 relancer la session

##Etape 4 : Tester la prédiction (et la sauvegarde)

In [36]:
import requests

donnees_employe = {
    "data": {
        "age": 35,
        "revenu_mensuel": 5000.0,
        "annees_dans_l_entreprise": 5,
        "annee_experience_totale": 10,
        "annees_dans_le_poste_actuel": 2,
        "annees_depuis_la_derniere_promotion": 1,
        "annees_sous_responsable_actuel": 3,
        "augmentation_salaire_precedente": 15.0,
        "nombre_experiences_precedentes": 2,
        "niveau_hierarchique_poste": 2,
        "note_evaluation_precedente": 3,
        "note_evaluation_actuelle": 3,
        "satisfaction_employee_environnement": 3,
        "satisfaction_employee_nature_travail": 3,
        "satisfaction_employee_equipe": 3,
        "satisfaction_employee_equilibre_pro_perso": 3,
        "nombre_participation_pee": 1,
        "nb_formations_suivies": 2,
        "distance_domicile_travail": 10,
        "niveau_education": 3
    }
}

reponse = requests.post("http://127.0.0.1:8000/predict", json=donnees_employe)
print("Réponse de l'API :", reponse.json())

INFO:     127.0.0.1:35426 - "POST /predict HTTP/1.1" 200 OK
Réponse de l'API : {'prediction': 'Non', 'probabilite': 0.18158140466034747}


In [37]:
import requests
import json

# 1. Test de la route /health
print("Test de la route /health :")
try:
    reponse_health = requests.get("http://127.0.0.1:8000/health")
    print("Statut :", reponse_health.status_code)
    print("Réponse :", reponse_health.json())
except Exception as e:
    print("Erreur lors de l'appel à /health :", e)

print("\n--- Test de la route /predict ---")
# 2. Test de la route /predict avec les données de l'employé structurées sous la clé 'data'
donnees_employe = {
    "data": {
        "age": 35,
        "revenu_mensuel": 5000.0,
        "annees_dans_l_entreprise": 5,
        "annee_experience_totale": 10,
        "annees_dans_le_poste_actuel": 2,
        "annees_depuis_la_derniere_promotion": 1,
        "annees_sous_responsable_actuel": 3,
        "augmentation_salaire_precedente": 15.0,
        "nombre_experiences_precedentes": 2,
        "niveau_hierarchique_poste": 2,
        "note_evaluation_precedente": 3,
        "note_evaluation_actuelle": 3,
        "satisfaction_employee_environnement": 3,
        "satisfaction_employee_nature_travail": 3,
        "satisfaction_employee_equipe": 3,
        "satisfaction_employee_equilibre_pro_perso": 3,
        "nombre_participation_pee": 1,
        "nb_formations_suivies": 2,
        "distance_domicile_travail": 10,
        "niveau_education": 3
    }
}

try:
    reponse_predict = requests.post("http://127.0.0.1:8000/predict", json=donnees_employe)
    print("Statut :", reponse_predict.status_code)
    try:
        print("Réponse :", reponse_predict.json())
    except json.JSONDecodeError:
        print("La réponse n'est pas au format JSON valide. Contenu de la réponse :")
        print(reponse_predict.text)
except Exception as e:
    print("Erreur lors de la requête :", e)

Test de la route /health :
INFO:     127.0.0.1:43002 - "GET /health HTTP/1.1" 200 OK
Statut : 200
Réponse : {'status': 'ok', 'modele_charge': True}

--- Test de la route /predict ---
INFO:     127.0.0.1:43018 - "POST /predict HTTP/1.1" 200 OK
Statut : 200
Réponse : {'prediction': 'Non', 'probabilite': 0.18158140466034747}


##Étape 5  Vérifier que la base de données a bien enregistré la prédiction

In [38]:
import sqlite3
import pandas as pd

# Connexion à la base de données SQLite
conn = sqlite3.connect("predictions.db")

# On lit la table
df = pd.read_sql("SELECT * FROM prediction_logs", conn)
print("Voici ce qui est enregistré dans la base de données :")
display(df)

conn.close()

Voici ce qui est enregistré dans la base de données :


,id,input_data,prediction,probability
0,1,"{""age"": 35, ""revenu_mensuel"": 5000.0}",Oui,0.555104
1,2,"{""age"": 35, ""revenu_mensuel"": 5000.0, ""annees_...",Non,0.181581
2,3,"{""age"": 35, ""revenu_mensuel"": 5000.0, ""annees_...",Non,0.181581
3,4,"{""age"": 35, ""revenu_mensuel"": 5000.0, ""annees_...",Non,0.181581
4,5,"{""age"": 35, ""revenu_mensuel"": 5000.0, ""annees_...",Non,0.181581


##Rapport de couverture

In [ ]:
!git clone https://github.com/antoinechemavonian/futurisys-api.git

In [39]:
%cd futurisys-api

/content/futurisys-api/futurisys-api


In [40]:
!pip install pytest-cov
!pytest --cov=app tests/test_api.py > rapport_couverture.txt


##Installe l'outil et générer le rapport

In [41]:
%cd futurisys-api

[Errno 2] No such file or directory: 'futurisys-api'
/content/futurisys-api/futurisys-api


In [42]:
# 1. On se place à la racine de Colab
%cd /content

# 2. On supprime l'ancien dossier s'il est corrompu
!rm -rf futurisys-api

# 3. On retélécharge le projet depuis GitHub
!git clone https://github.com/antoinechemavonian/futurisys-api.git

# 4. On entre dans le dossier
%cd futurisys-api

# 5. On vérifie que le dossier tests est bien là
!ls tests

# 6. On installe les outils
!pip install pytest pytest-cov

# 7. On lance le test et on l'affiche à l'écran
!pytest --cov=app tests/test_api.py

# 8. On sauvegarde le résultat dans le fichier texte
!pytest --cov=app tests/test_api.py > rapport_couverture.txt

/content
Cloning into 'futurisys-api'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 94 (delta 35), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 1.76 MiB | 3.86 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/futurisys-api
__init__.py  test_api.py
============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0
rootdir: /content/futurisys-api
plugins: cov-7.1.0, platformdirs-4.14.3, langsmith-0.14.17, anyio-4.16.0, typeguard-4.6.0
collected 2 items                                                              

tests/test_api.py ..                                                     [100%]

================================ tests coverage ================================
_______________ coverage: platform linux, python 3.13.18-final-0 _______________

Name            

##

##